# 第 22 章 勾配ブースティング

材料の配合と試験する材齢から，圧縮強度を予測します。平均から始めて残差を小さな木で補い，反復数と検証の誤差を比べます。講義部分を上から実行し，後半の約 20 分では別の検証行を選んで更新を調べます。

- 現在の予測・残差・木の出力から，一段後の予測を計算する。
- 一行の変化と全体の MAE を区別し，森との比較から候補を判断する。

講義本文：Book 第 22 章（授業で案内する教材） / [配布データ](https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/concrete-strength.csv)

**使い方**：Colab では自分のドライブへコピーし，ローカルではこのノートを保存してから実行してください。結果だけでなく予想と報告も残します。提出前に上から全セルを再実行し，保存します。保留行の成績はこのノートでは計算しません。

## 四行の残差

この四行は仕組みのために作った架空例です。初期予測を強度の平均とします。平均 6 から見て，どの行で上向き・下向きの補正が必要でしょうか。残差は「実測−予測」です。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

toy = pd.DataFrame({
    "age": [7, 14, 28, 56],
    "strength": [2.0, 4.0, 8.0, 10.0],
}, index=["A", "B", "C", "D"])
toy["initial"] = toy["strength"].mean()
toy["residual_0"] = toy["strength"] - toy["initial"]
toy

## 残差の木を一つ加える

正解として強度ではなく `residual_0` を渡します。材齢 21 日以下で分かれる木の左右の葉は，残差の平均 −3 と +3 を返します。学習率 0.5 なら，A の新しい予測をいくつにするか，実行前に計算してください。

In [ ]:
residual_tree = DecisionTreeRegressor(max_depth=1, random_state=2026)
residual_tree.fit(toy[["age"]], toy["residual_0"])
toy["tree_1"] = residual_tree.predict(toy[["age"]])

learning_rate = 0.5
toy["prediction_1"] = toy["initial"] + learning_rate * toy["tree_1"]
toy["residual_1"] = toy["strength"] - toy["prediction_1"]
toy[["strength", "initial", "residual_0", "tree_1",
     "prediction_1", "residual_1"]]

## 新しい残差で次の木を学ぶ

新しい残差は −2.5，−0.5，+0.5，+2.5 です。次の木はこの値を学びます。入力の材齢は同じです。二本目の葉の出力と，半分を加えた予測を先に計算してみましょう。学習率 1 なら一段で 3，3，9，9 になりますが，小さい学習率なら同じ段数で必ずよいわけではありません。

In [ ]:
second_tree = DecisionTreeRegressor(max_depth=1, random_state=2026)
second_tree.fit(toy[["age"]], toy["residual_1"])
toy["tree_2"] = second_tree.predict(toy[["age"]])
toy["prediction_2"] = toy["prediction_1"] + learning_rate * toy["tree_2"]
toy[["strength", "prediction_1", "residual_1", "tree_2", "prediction_2"]]

In [ ]:
fig, axes = plt.subplots(1, 3, sharey=True, layout="constrained", figsize=(8, 3.4))
for ax, column, title in zip(axes, ["initial", "prediction_1", "prediction_2"],
                             ["Initial mean", "After 1 tree", "After 2 trees"]):
    positions = np.arange(len(toy))
    ax.vlines(positions, toy[column], toy["strength"], color="#aab7bd")
    ax.scatter(positions, toy["strength"], color="#286b70", label="Observed", zorder=3)
    ax.scatter(positions, toy[column], color="#a34f30", marker="x", label="Predicted", zorder=4)
    ax.set(title=title, xticks=positions, xticklabels=toy.index, xlabel="Teaching example row")
    ax.grid(axis="y", color="#e3e8eb")
axes[0].set_ylabel("Strength (MPa)")
axes[0].legend(frameon=False)
plt.show()

図の点は架空の実測，×は予測です。二段目で A・D の誤差が減り，B・C の予測が正解を越えるところを読んでください。木がまとめる残差は葉の平均なので，個々の残差をそのまま足しているわけではありません。

## 学習率を変えると次の木も変わる

最初の木を全部足すと予測は 3・3・9・9，残差は −1・+1・−1・+1 です。半分を足した場合の残差と比べて，どの行で符号が違いますか。同じ側へ補正したい行のまとまり方が変わると，次の木の分岐も変わり得ます。

In [ ]:
full_rate_prediction = toy["initial"] + toy["tree_1"]
full_rate_residual = toy["strength"] - full_rate_prediction
full_rate_second_tree = DecisionTreeRegressor(max_depth=1, random_state=2026)
full_rate_second_tree.fit(toy[["age"]], full_rate_residual)

pd.DataFrame({
    "residual_at_0.5": toy["residual_1"],
    "residual_at_1.0": full_rate_residual,
    "next_tree_at_0.5": toy["tree_2"],
    "next_tree_at_1.0": full_rate_second_tree.predict(toy[["age"]]),
}).round(3)

学習率 1 の場合，次の木は A だけを左へ分け，A へ −1，B・C・D へ平均 1/3 を返しました。0.5 では A・B へ −1.5，C・D へ +1.5 です。学習率は，次の木へ渡す正解にも影響します。完成した予測を最後に半分にすることと，一段ごとに半分ずつ足して学ぶことが違う理由を説明してください。

## 圧縮強度のデータ

材料 7 列と材齢を入力し，強度（MPa）を予測します。学習 631 行・256 配合と検証 208 行・86 配合を使い，同じ配合が両側へまたがらない分割を保持します。第 20 章から同じ検証を候補比較に再利用しています。

次のセルは，プロジェクト直下，`practice` フォルダー，CSV を置いた作業フォルダーの順にファイルを探します。見つからなければ公開 CSV を読みます。管理用 ID と `split` は入力に入れません。

In [ ]:
from pathlib import Path

candidates = [Path("data/concrete-strength.csv"),
              Path("../data/concrete-strength.csv"),
              Path("concrete-strength.csv")]
source = next((path for path in candidates if path.exists()),
              "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/concrete-strength.csv")
concrete = pd.read_csv(source)
features = [
    "cement", "slag", "fly_ash", "water", "superplasticizer",
    "coarse_aggregate", "fine_aggregate", "age",
]
train = concrete.loc[concrete["split"] == "train"].copy()
valid = concrete.loc[concrete["split"] == "validation"].copy()
X_train, y_train = train[features], train["strength"]
X_valid, y_valid = valid[features], valid["strength"]

## 学習率と木の数を指定する

二乗誤差で，深さ 2 の木を 300 段，学習率 0.05 で加えます。`fit` の正解に渡すのは学習側だけです。検証側の残差を木に学ばせてはいけません。

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

boosting = GradientBoostingRegressor(
    loss="squared_error",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=2,
    random_state=2026,
)
boosting.fit(X_train, y_train)

## 最初の補正を照合する

検証の元行番号 1 を見ます。最初の木の出力は約 +23.39 MPa です。これは強度の予測そのものではなく，学習率を掛けて加える補正です。学習平均約 36.68 にどれだけ足すことになるでしょうか。

`estimators_[0, 0]` の一つ目の 0 で第一段を選び，二つ目の 0 でその段にある一本の木を選びます。

In [ ]:
example = valid.loc[valid["record_id"] == 1]
initial_mean = y_train.mean()
first_tree = boosting.estimators_[0, 0]
first_correction = first_tree.predict(example[features].to_numpy())[0]
first_prediction = initial_mean + boosting.learning_rate * first_correction
print(f"Initial mean: {initial_mean:.4f} MPa")
print(f"First tree output: {first_correction:.4f} MPa")
print(f"Amount added: {boosting.learning_rate * first_correction:.4f} MPa")
print(f"Prediction after one tree: {first_prediction:.4f} MPa")

## 一行の予測の推移

`staged_predict()` は，木を一段加えるごとの予測を順に返します。リストの 0 番目が一段目で，その中に検証の全行分の予測が入力と同じ順に並びます。

`np.flatnonzero()` で `record_id == 1` が成り立つ位置を探し，各段の予測からその位置の一行分を取り出します。予測は上向きに動きますが，実測の約 79.99 MPa まで届くかを確かめます。

In [ ]:
valid_predictions = list(boosting.staged_predict(X_valid))
train_predictions = list(boosting.staged_predict(X_train))
example_position = np.flatnonzero(valid["record_id"].to_numpy() == 1)[0]
stages_to_read = [1, 10, 100, 300]
example_history = pd.DataFrame({
    "stage": stages_to_read,
    "prediction": [valid_predictions[stage - 1][example_position]
                   for stage in stages_to_read],
})
example_history["actual"] = example["strength"].iloc[0]
example_history["residual"] = example_history["actual"] - example_history["prediction"]
example_history.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, sharey=True, layout="constrained", figsize=(8, 3.4))
for ax, record in zip(axes, [1, 11]):
    position = np.flatnonzero(valid["record_id"].to_numpy() == record)[0]
    trajectory = [initial_mean] + [pred[position] for pred in valid_predictions]
    actual = y_valid.iloc[position]
    ax.plot(range(301), trajectory, color="#a34f30", label="Predicted")
    ax.axhline(actual, color="#286b70", linestyle="--", label="Observed")
    ax.set(title=f"Record {record}", xlabel="Number of added trees", ylim=(30, 85))
    ax.grid(color="#e3e8eb")
axes[0].set_ylabel("Strength (MPa)")
axes[0].legend(frameon=False)
plt.show()

左は行 1，右は行 11 です。破線は各行の実測です。行 11 では途中で実測を越えます。すべての検証行が毎回近づく更新ではありません。検証の実測は，予測後の誤差の確認にだけ使います。

## 学習と検証の MAE

すべての段で同じ MAE を計算します。学習損失の `train_score_` を MAE として使うことはしません。初めの 100 段と，その後の 200 段で，改善の大きさはどう違うでしょうか。

In [ ]:
history = pd.DataFrame({
    "stage": range(1, 301),
    "train_mae": [mean_absolute_error(y_train, pred) for pred in train_predictions],
    "valid_mae": [mean_absolute_error(y_valid, pred) for pred in valid_predictions],
}).set_index("stage")
history.loc[[1, 10, 30, 50, 100, 200, 300]].round(3)

In [ ]:
fig, ax = plt.subplots(layout="constrained", figsize=(7.2, 3.8))
ax.plot(history.index, history["train_mae"], color="#286b70", label="Train")
ax.plot(history.index, history["valid_mae"], color="#a34f30", label="Validation")
ax.set(xlabel="Number of added trees", ylabel="MAE (MPa)", xlim=(0, 305), ylim=(0, 15))
ax.grid(color="#e3e8eb")
ax.legend(frameon=False)
plt.show()

## 全体が改善しても遠ざかる一行

100 段から 300 段へ，どのくらいの行で絶対誤差が減り，増えたでしょうか。元行 82 は材齢 3 日，実測約 25.20 MPa の記録です。予測はすでに実測より大きいのですが，追加の補正でどう動いたかを読みます。

In [ ]:
stage_comparison = valid[["record_id", "strength"]].copy()
stage_comparison["prediction_100"] = valid_predictions[99]
stage_comparison["prediction_300"] = valid_predictions[299]
stage_comparison["error_100"] = abs(stage_comparison["strength"] - stage_comparison["prediction_100"])
stage_comparison["error_300"] = abs(stage_comparison["strength"] - stage_comparison["prediction_300"])
change_in_error = stage_comparison["error_300"] - stage_comparison["error_100"]
print("Smaller absolute error:", (change_in_error < 0).sum())
print("Larger absolute error: ", (change_in_error > 0).sum())
stage_comparison.loc[stage_comparison["record_id"] == 82].round(3)

133 行で改善し，75 行で悪化しました。82 番は約 33.28 から 34.68 MPa へ上がり，遠ざかります。途中の 102 段目で同じ葉へ入った学習行を調べます。この木ではセメント量が約 171.34，高炉スラグが約 145.15 をともに超える葉です。下の補助コードの `apply()` は各行が到達した葉の番号を返すので，同じ番号の学習行を集められます。

In [ ]:
step_tree = boosting.estimators_[101, 0]
row_82 = valid.loc[valid["record_id"] == 82]
position_82 = np.flatnonzero(valid["record_id"].to_numpy() == 82)[0]
target_leaf = step_tree.apply(row_82[features].to_numpy())[0]
in_same_leaf = step_tree.apply(X_train.to_numpy()) == target_leaf
residual_before_step = y_train.to_numpy() - train_predictions[100]

pd.Series({
    "training_rows_in_leaf": in_same_leaf.sum(),
    "mean_training_residual": residual_before_step[in_same_leaf].mean(),
    "tree_output": step_tree.predict(row_82[features].to_numpy())[0],
    "validation_residual_before": y_valid.iloc[position_82] - valid_predictions[100][position_82],
    "prediction_before": valid_predictions[100][position_82],
    "prediction_after": valid_predictions[101][position_82],
}).round(4)

この葉に入った学習側の 100 行では残差平均が約 +3.005 MPa で，上げる補正が必要でした。0.05 倍の約 +0.1502 を加えます。しかし 82 番の検証行では実測との差が約 −8.0590 で，下げたい方向でした。学習側の補正がこの検証一行には役立たなかった理由を説明してください。検証の正解を使ってこの木を直すと，学習と分けた行への評価ではなくなります。

## 最小値と判断

検証の最小は 299 段ですが，300 段との差は約 0.00066 MPa です。この表での最小と，ほかの配合でも安定して優れることは違います。比較は事前に決めた 300 段で続けます。300 段より先の成績や，別の学習率で必要な反復数はこの曲線から決まりません。

In [ ]:
best_stage = int(history["valid_mae"].idxmin())
print(f"Smallest validation MAE within 1-300: stage {best_stage}")
history.loc[[best_stage, 300]].round(6)

## 森と比べる

前章と同じ 200 本・特徴量候補 0.7 の森を，同じ入力，同じ学習行で作ります。森は強度を学ぶ木の予測を平均し，こちらは残差を学ぶ木の補正を順に足します。どちらの検証 MAE が小さいでしょうか。

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest = RandomForestRegressor(
    n_estimators=200, max_features=0.7, random_state=2026, n_jobs=2,
)
forest.fit(X_train, y_train)
forest_prediction = forest.predict(X_valid)
comparison = pd.DataFrame({
    "method": ["Training mean", "Boosting: 100", "Boosting: 300", "Forest: 200"],
    "validation_mae": [
        mean_absolute_error(y_valid, np.full(len(y_valid), initial_mean)),
        history.loc[100, "valid_mae"],
        history.loc[300, "valid_mae"],
        mean_absolute_error(y_valid, forest_prediction),
    ],
}).set_index("method")
comparison.round(3)

In [ ]:
row_comparison = valid[["record_id", "age", "strength"]].copy()
row_comparison["boosting"] = valid_predictions[-1]
row_comparison["forest"] = forest_prediction
row_comparison["boosting_abs_error"] = abs(row_comparison["strength"] - row_comparison["boosting"])
row_comparison["forest_abs_error"] = abs(row_comparison["strength"] - row_comparison["forest"])
row_comparison.loc[row_comparison["record_id"].isin([1, 11])].round(3)

全体では森の MAE が小さい一方，ここで表示した二行ではブースティングの方が近い結果です。数行を読む目的と，全体で候補を選ぶ目的を分けます。

## 選択実習：約 20 分

**問い：全体で役立った追加の補正は，自分が選んだ一行でも役立つか。**

1. 次の候補から元行番号を一つ，反復数を 30・100・300 段から二つ選びます（約 5 分）。選んだ理由と予想を，結果を出す前に書いてください。
2. 選択欄と一段更新の式を埋めます。選んだ行の更新と誤差を，全検証 MAE と並べます（約 9 分）。
3. その行と全体を分けて，3〜4 文で報告します（約 6 分）。

候補の強度は検証後の読解に使います。これを残差の学習へ渡したり，保留を見て選んだりしません。

In [ ]:
candidate_records = [5, 12, 58, 93, 105, 128]
valid.loc[valid["record_id"].isin(candidate_records),
          ["record_id", "recipe_id", "age", "strength"]].round(3)

### 選んだ理由と予想

- 選ぶ元行番号と理由：
- 比較する二つの反復数と理由：
- 初期平均から上向き・下向きのどちらが役立ちそうか：
- 全体での改善と同じ変化がこの行にも見えそうか：

In [ ]:
selected_record = None  # candidate_records から一つ選ぶ
earlier_stage = None
later_stage = None

`earlier_stage` と `later_stage` は 30・100・300 のうち，小さい方と大きい方を指定します。選択を変えたら，選択欄のコードセルから下を順に実行します。次の出力の元行番号を見て，最初の木がどの行に答えたかを確かめてください。

In [ ]:
def practice_row_position(record_id, first_stage, second_stage):
    if record_id is None or first_stage is None or second_stage is None:
        return None
    if record_id not in candidate_records:
        raise ValueError("元行番号は候補の中から選んでください。")
    allowed_stages = [30, 100, 300]
    if first_stage not in allowed_stages or second_stage not in allowed_stages or first_stage >= second_stage:
        raise ValueError("30・100・300から，小さい段と大きい段を選んでください。")
    return np.flatnonzero(valid["record_id"].to_numpy() == record_id)[0]

selected_position = practice_row_position(selected_record, earlier_stage, later_stage)
if selected_position is None:
    print("元行番号と二つの反復数を記入してください。")
else:
    selected_input = X_valid.iloc[[selected_position]]
    selected_correction = first_tree.predict(selected_input.to_numpy())[0]
    print(f"Record ID: {valid['record_id'].iloc[selected_position]}")
    print(f"Initial mean: {initial_mean:.4f} MPa")
    print(f"First tree output: {selected_correction:.4f} MPa")
    print(f"Learning rate: {boosting.learning_rate}")

### 一段の計算

`one_step_prediction` の `None` を，新しい予測を計算する式へ置き換えます。使う値は `initial_mean`，`boosting.learning_rate`，`selected_correction` です。元行番号を確認し，モデルの一段目と自分の計算を照合してください。負の補正なら足す量も負です。

In [ ]:
selected_position = practice_row_position(selected_record, earlier_stage, later_stage)
if selected_position is None:
    print("選択欄を記入してください。")
else:
    selected_input = X_valid.iloc[[selected_position]]
    selected_correction = first_tree.predict(selected_input.to_numpy())[0]
    one_step_prediction = None  # 一段の更新を表す式に置き換える
    if one_step_prediction is None:
        print("一段の更新式を記入してください。")
    else:
        print(f"Record ID: {valid['record_id'].iloc[selected_position]}")
        display(pd.Series({
            "your_calculation": one_step_prediction,
            "model_stage_1": valid_predictions[0][selected_position],
            "difference": one_step_prediction - valid_predictions[0][selected_position],
        }).round(6))

### 個別の変化と全体の MAE

二つの反復数では，予測がどちらへ動き，選んだ行の絶対誤差がどう変わったでしょうか。`all_validation_mae` は選んだ一行だけでなく，同じ 208 行全体で計算した値です。最後に森の結果も並べます。

In [ ]:
exercise_result = None
selected_position = practice_row_position(selected_record, earlier_stage, later_stage)
if selected_position is None:
    print("選択欄を記入してから実行してください。")
else:
    actual = y_valid.iloc[selected_position]
    exercise_rows = []
    for stage in [earlier_stage, later_stage]:
        prediction = valid_predictions[stage - 1][selected_position]
        exercise_rows.append({
            "method": f"Boosting: {stage}", "actual": actual,
            "prediction": prediction, "residual": actual - prediction,
            "row_abs_error": abs(actual - prediction),
            "all_train_mae": history.loc[stage, "train_mae"],
            "all_validation_mae": history.loc[stage, "valid_mae"],
        })
    prediction = forest_prediction[selected_position]
    exercise_rows.append({
        "method": "Forest: 200", "actual": actual,
        "prediction": prediction, "residual": actual - prediction,
        "row_abs_error": abs(actual - prediction),
        "all_train_mae": mean_absolute_error(y_train, forest.predict(X_train)),
        "all_validation_mae": mean_absolute_error(y_valid, forest_prediction),
    })
    exercise_result = pd.DataFrame(exercise_rows).set_index("method")
    print(f"Record ID: {valid['record_id'].iloc[selected_position]}")
    display(exercise_result.round(3))

### 報告：3〜4 文

- 選んだ行と二つの段で，予測・絶対誤差はどう変わったか。予想をどう更新したか。
- 一段目の補正はどちら向きで，何を学んだ木の出力だったか。
- 一行の結果と全検証 MAE を区別すると，どの方法を候補にするか。
- 同じ検証の再利用を踏まえて，次に何を確かめたいか。

ここに報告を書いてください：


### 提出前の確認

選択欄と更新式を埋め，予想・更新の照合・結果の表・報告を残します。上から全セルを再実行し，表示された元行番号と報告の番号を照合してから保存してください。余裕があれば，違う一行で同じ二段を比較し，個別の結果が違う理由を考えます。保留の評価は行いません。

### 出典

I-Cheng Yeh, *Concrete Compressive Strength*, UCI Machine Learning Repository, [DOI: 10.24432/C5PK67](https://doi.org/10.24432/C5PK67), CC BY 4.0。配布 CSV は元表の精度を保ち，列名・管理列・配合群での分割を追加しています。[加工と分割の記録](https://github.com/sohei-takagi/python-ds-course-materials/blob/main/data/concrete-sources.md)。

[scikit-learn 1.8：GradientBoostingRegressor](https://scikit-learn.org/1.8/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html) / [Ensemble methods](https://scikit-learn.org/1.8/modules/ensemble.html#gradient-boosting)